# Stellan Lite and Max · SWE-bench Verified pilot
Internet ON, GPU T4 x2. Save Version → Save & Run All. Ten identical tasks, one answer per task/model.
This is a retrieved-context model baseline, not the full Stick AI agent. Kaggle generates predictions; score separately with the official Docker harness. See SWE_BENCH.md.


In [ ]:
%%bash
set -euo pipefail
LLAMA_COMMIT=cd959bae99ff4bed22bef2604f45241c09bbb8c8
WORK=/tmp/stick-swe
mkdir -p "$WORK"
cd "$WORK"
GPU=0
command -v nvidia-smi >/dev/null && nvidia-smi -L >/dev/null 2>&1 && GPU=1
[ "$GPU" = 1 ] || { echo 'Select GPU T4 x2 before running this notebook.'; exit 1; }
echo "== Node.js"
curl -fsSL https://nodejs.org/dist/v24.22.1/node-v24.22.1-linux-x64.tar.xz | tar -xJ
export PATH="$WORK/node-v24.22.1-linux-x64/bin:$PATH"; node --version

echo "== Stick AI's benchmark script"
rm -rf stick_ai
git init -q stick_ai
git -C stick_ai remote add origin https://github.com/alfredmurray-goat/stick_ai.git
git -C stick_ai fetch -q --depth 1 origin a7f8d3cbd0a4a61a2eac992072e44fa89ebe0fa2
git -C stick_ai checkout -q --detach FETCH_HEAD
(cd stick_ai && npm install --ignore-scripts --no-audit --no-fund --loglevel=error && npm run build --silent)

echo "== llama.cpp at the app's pinned version ($([ $GPU = 1 ] && echo 'with CUDA, about 20 minutes' || echo 'about 10 minutes'))"
CUDA_FLAGS=(-DGGML_CUDA=OFF)
BUILD_MODE=cpu
if [ "$GPU" = 1 ]; then
  CUDA_FLAGS=(-DGGML_CUDA=ON -DGGML_CUDA_NO_VMM=ON -DCMAKE_CUDA_ARCHITECTURES=75)
  BUILD_MODE=cuda-sm75-no-vmm
  command -v nvcc >/dev/null || { echo "CUDA compiler missing: select GPU T4 x2 and restart the Kaggle session."; exit 1; }
fi
PATCH_SHA=$(sha256sum stick_ai/scripts/llama-split-dir.patch | cut -c1-12)
BUILD_DIR="llama-build-${LLAMA_COMMIT:0:12}-${PATCH_SHA}-${BUILD_MODE}"
SERVER="$WORK/$BUILD_DIR/bin/llama-server"
if [ ! -x "$SERVER" ]; then
  rm -rf llama.cpp "$BUILD_DIR" && git init -q llama.cpp && git -C llama.cpp remote add origin https://github.com/ggml-org/llama.cpp.git
  git -C llama.cpp fetch -q --depth 1 origin "$LLAMA_COMMIT" && git -C llama.cpp checkout -q --detach FETCH_HEAD
  [ "$(git -C llama.cpp rev-parse HEAD)" = "$LLAMA_COMMIT" ] || { echo "The inference source could not be verified; rerun the notebook."; exit 1; }
  git -C llama.cpp apply "$WORK/stick_ai/scripts/llama-split-dir.patch"
  cmake -S llama.cpp -B "$BUILD_DIR" -DCMAKE_BUILD_TYPE=Release -DBUILD_SHARED_LIBS=OFF -DGGML_NATIVE=OFF -DGGML_OPENMP=OFF -DGGML_METAL=OFF -DGGML_BLAS=OFF -DLLAMA_OPENSSL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF -DLLAMA_BUILD_SERVER=ON -DLLAMA_USE_PREBUILT_UI=OFF -DLLAMA_BUILD_UI=OFF "${CUDA_FLAGS[@]}" > cmake.log 2>&1 || { cat cmake.log; exit 1; }
  cmake --build "$BUILD_DIR" --config Release --target llama-server -j "$(nproc)" > build.log 2>&1 || { tail -30 build.log; exit 1; }
fi
if [ "$GPU" = 1 ]; then
  "$SERVER" --list-devices > devices.log 2>&1 || { cat devices.log; exit 1; }
  cat devices.log
  grep -Eq 'CUDA[0-9]+:' devices.log || { echo "The server has no CUDA devices; restart with GPU T4 x2 instead of continuing on the CPU."; exit 1; }
fi

python3 -m pip -q install datasets huggingface_hub
cd stick_ai
OUT=/kaggle/working
TASKS="$OUT/swe-tasks.jsonl"
[ -f "$TASKS" ] || python3 scripts/benchmark-swe.py --prepare --tasks "$TASKS" --limit 10 --seed 42
python3 -m pip freeze > "$OUT/swe-python-packages.txt"
git rev-parse HEAD > "$OUT/swe-stick-commit.txt"
cp "$WORK/devices.log" "$OUT/swe-devices.log"
server_pid=''
cleanup() { [ -z "$server_pid" ] || { kill "$server_pid" 2>/dev/null || true; wait "$server_pid" 2>/dev/null || true; }; }
trap cleanup EXIT
for MODEL in stellan-1-lite stellan-1-max; do
  if [ "$MODEL" = stellan-1-lite ]; then
    URL=https://huggingface.co/OfficialSejeGed/stellan-1-lite-gguf/resolve/94f4d2ffd0990289cda48d9505844f90ac6e7144/stellan-1-lite-q4_k_m.gguf
    SHA=84328c452ddef6122b35702f754935ebd7c8bf95a8a9f2b718392cc68968453e
    EXTRA=(--no-jinja --skip-chat-parsing --chat-template mistral-v1)
  else
    read -r URL SHA < <(node -e 'import("./dist/portable/max-release.js").then(m => console.log(m.maxRelease.url + " " + m.bundledMax.sha256))')
    EXTRA=()
  fi
  GGUF="$WORK/$MODEL.gguf"
  [ -f "$GGUF" ] || curl -fL --retry 5 -o "$GGUF" "$URL"
  echo "$SHA  $GGUF" | sha256sum -c -
  RUN="$OUT/swe-$MODEL"
  mkdir -p "$RUN"
  ARGS=(--model "$GGUF" --alias "$MODEL" --ctx-size 8192 --host 127.0.0.1 --port 18100 --no-webui --no-warmup --gpu-layers 999 "${EXTRA[@]}")
  printf '%s\n' "${ARGS[@]}" > "$RUN/server-args.txt"
  "$SERVER" "${ARGS[@]}" > "$RUN/server.log" 2>&1 & server_pid=$!
  ready=0
  for attempt in $(seq 1 240); do
    kill -0 "$server_pid" 2>/dev/null || { tail -40 "$RUN/server.log"; exit 1; }
    if curl -fsS http://127.0.0.1:18100/health >/dev/null 2>&1; then ready=1; break; fi
    sleep 2
  done
  [ "$ready" = 1 ] || { echo 'Server startup timed out'; exit 1; }
  timeout 10h python3 scripts/benchmark-swe.py --tasks "$TASKS" --endpoint http://127.0.0.1:18100 --model "$MODEL" --server-bin "$SERVER" --gguf "$GGUF" --out "$RUN" 2>&1 | tee "$RUN/inference.log"
  cleanup
  server_pid=''
  rm "$GGUF"
done
echo 'Predictions saved. Download Output and grade with the official Docker harness. No score yet.'
